# visualization_008

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (222).ipynb`

| Field | Value |
|---|---|
| Section | `visualization` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 9 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df1 = pd.read_csv('/content/drive/MyDrive/Test_Tradebook/merged_output.csv')
df1

In [ ]:
df2 = pd.read_csv('/content/drive/MyDrive/Test_Tradebook/2025/tradebook_TOP5_25.csv')
df2

In [ ]:
import pandas as pd

# load your two tradebooks
tb1 = df1  # 2022-01-03 → 2025-04-30
tb2 = df2  # 2025-01-01 → 2025-06-30

# convert Entry Time to datetime for filtering
tb1["Entry Time"] = pd.to_datetime(tb1["Entry Time"])
tb2["Entry Time"] = pd.to_datetime(tb2["Entry Time"])

# remove overlap: keep tb2 trades strictly after tb1 max date
cutoff_date = tb1["Entry Time"].max()
tb2_filtered = tb2[tb2["Entry Time"] > cutoff_date]

# combine tradebooks
merged = pd.concat([tb1, tb2_filtered], ignore_index=True)

# sort just in case
merged = merged.sort_values("Entry Time").reset_index(drop=True)

print(merged["Entry Time"].min(), "→", merged["Entry Time"].max())
print(len(merged), "rows")
merged

In [ ]:
merged['Ticker'].unique()

In [ ]:
import pandas as pd

# assume merged is your dataset after union - intersection

# 1. sort by Ticker and Entry Time
merged = merged.sort_values(by=["Ticker", "Entry Time"]).reset_index(drop=True)

# 2. recompute cumulative PnL per Ticker
merged["Cumulative PnL"] = merged.groupby("Ticker")["PnL"].cumsum()

merged

In [ ]:
# remove last 4 rows
merged = merged.iloc[:-4].reset_index(drop=True)

merged

In [ ]:
import plotly.express as px

# merged = your final combined tradebook DataFrame
fig = px.scatter(
    merged,
    x="Entry Time",
    y=merged.groupby("Ticker").cumcount(),  # trade index per ticker
    color="Ticker",
    facet_row="Ticker",
    title="Tradebook Timeline by Ticker",
    height=800
)

fig.update_traces(marker=dict(size=4, opacity=0.6))
fig.update_layout(
    showlegend=True,
    xaxis_title="Entry Time",
    yaxis_title="Trade Index"
)

fig.show()


In [ ]:
merged.to_csv('/content/drive/MyDrive/Test_Tradebook.csv')